In [1]:
import pandas as pd
import matplotlib.pyplot as plt
print("pandas", pd.__version__)

Matplotlib is building the font cache; this may take a moment.


pandas 3.1.0


In [3]:
import sys
print(sys.executable)


/Users/terry/Documents/ni-ireland-inactivity/.venv/bin/python


In [6]:
import os
print(os.listdir("../data/raw"))


['women-in-NI-headline-tables-2023.XLSX', 'labour-market-structure-2025.xlsx', '.gitkeep', 'industry-2025.xlsx', 'Labour Force Survey - Tables for Local Government Districts 2009 to 2025.xlsx']


In [8]:
path = "../data/raw/Labour Force Survey - Tables for Local Government Districts 2009 to 2025.xlsx"
sheets = pd.read_excel(path, sheet_name=None, header=None)

rows = []
for name, df in sheets.items():
    if not name.isdigit():
        continue
    ni_rows = df[df[0].str.strip() == "Northern Ireland"]
    activity = ni_rows.iloc[-1, 2]
    rows.append({"year": int(name), "inactivity": 100 - activity})

ni = pd.DataFrame(rows).sort_values("year")
print(ni)

    year  inactivity
16  2009        30.1
15  2010        29.1
14  2011        27.6
13  2012        27.6
12  2013        27.5
11  2014        27.6
10  2015        27.1
9   2016        26.0
8   2017        27.6
7   2018        27.2
6   2019        26.2
5   2020        28.0
4   2021        27.0
3   2022        26.3
2   2023        24.7
1   2024        24.3
0   2025        24.1


In [10]:
path_ie = "../data/raw/LRAC64TTIEA156N.csv"
ie = pd.read_csv(path_ie)
print(ie.head())

  observation_date  LRAC64TTIEA156N
0       2000-01-01           68.175
1       2001-01-01           68.550
2       2002-01-01           68.550
3       2003-01-01           68.825
4       2004-01-01           69.500


In [11]:
ie["year"] = ie["observation_date"].str[:4].astype(int)
ie["inactivity"] = 100 - ie["LRAC64TTIEA156N"]
ie = ie[(ie["year"] >= 2009) & (ie["year"] <= 2025)]
ie = ie[["year", "inactivity"]]

both = ni.merge(ie, on="year", suffixes=("_ni", "_ie"))
both["gap"] = both["inactivity_ni"] - both["inactivity_ie"]
print(both.round(1))

    year  inactivity_ni  inactivity_ie  gap
0   2009           30.1           27.0  3.1
1   2010           29.1           28.4  0.6
2   2011           27.6           28.8 -1.2
3   2012           27.6           28.9 -1.3
4   2013           27.5           28.2 -0.7
5   2014           27.6           28.2 -0.6
6   2015           27.1           28.0 -0.9
7   2016           26.0           27.4 -1.4
8   2017           27.6           27.4  0.2
9   2018           27.2           27.2 -0.0
10  2019           26.2           26.8 -0.6
11  2020           28.0           29.2 -1.2
12  2021           27.0           25.5  1.5
13  2022           26.3           23.2  3.0
14  2023           24.7           22.5  2.2
15  2024           24.3           22.1  2.2
16  2025           24.1           21.6  2.5


In [13]:
ni.to_csv("../data/clean/ni_inactivity_2009_2025.csv", index=False)
ie.to_csv("../data/clean/ie_inactivity_2009_2025.csv", index=False)

In [14]:
both.to_csv("../data/clean/ni_ie_inactivity_2009_2025.csv", index=False)

In [15]:
path_sex = "../data/raw/women-in-NI-headline-tables-2023.XLSX"
df = pd.read_excel(path_sex, sheet_name="Figure 7", header=7)
print(df.columns)

ni_sex = df.iloc[:, [0, 1, 4]]
ni_sex.columns = ["year", "ni_men", "ni_women"]
ni_sex[["ni_men", "ni_women"]] = ni_sex[["ni_men", "ni_women"]] * 100
print(ni_sex)

Index(['Year', 'Male Economically Inactive rate\n[Note 5]\n(%)',
       'Male Economically Inactive rate lower limit\n(%)',
       'Male Economically Inactive rate upper limit\n(%)',
       'Female Economically Inactive rate\n[Note 5]\n(%)',
       'Female Economically Inactive rate lower limit\n(%)',
       'Female Economically Inactive rate upper limit\n(%)'],
      dtype='str')
    year  ni_men  ni_women
0   2013    21.8      33.9
1   2014    21.6      33.7
2   2015    21.1      33.3
3   2016    20.6      33.0
4   2017    21.1      31.8
5   2018    23.6      31.9
6   2019    21.7      30.6
7   2020    21.2      30.7
8   2021    23.9      31.5
9   2022    24.4      30.0
10  2023    21.5      29.4


In [16]:
path_ie_women = "../data/raw/LRAC64FEIEA156N.csv"
path_ie_men = "../data/raw/LRAC64MAIEA156N.csv"

ie_w = pd.read_csv(path_ie_women)
ie_w["year"] = ie_w["observation_date"].str[:4].astype(int)
ie_w["ie_women"] = 100 - ie_w["LRAC64FEIEA156N"]
ie_w = ie_w[(ie_w["year"] >= 2013) & (ie_w["year"] <= 2023)]
ie_w = ie_w[["year", "ie_women"]]

ie_m = pd.read_csv(path_ie_men)
ie_m["year"] = ie_m["observation_date"].str[:4].astype(int)
ie_m["ie_men"] = 100 - ie_m["LRAC64MAIEA156N"]
ie_m = ie_m[(ie_m["year"] >= 2013) & (ie_m["year"] <= 2023)]
ie_m = ie_m[["year", "ie_men"]]

sex = ni_sex.merge(ie_w, on="year").merge(ie_m, on="year")
print(sex.round(1))

    year  ni_men  ni_women  ie_women  ie_men
0   2013    21.8      33.9      34.7    21.7
1   2014    21.6      33.7      34.9    21.3
2   2015    21.1      33.3      34.9    21.0
3   2016    20.6      33.0      33.8    20.8
4   2017    21.1      31.8      33.5    21.2
5   2018    23.6      31.9      33.1    21.2
6   2019    21.7      30.6      32.8    20.8
7   2020    21.2      30.7      35.2    23.2
8   2021    23.9      31.5      30.2    20.6
9   2022    24.4      30.0      28.2    18.2
10  2023    21.5      29.4      27.0    18.0


In [17]:
sex.to_csv("../data/clean/inactivity_by_sex_2013_2023.csv", index=False)